# CRM Access Governance & Customer Data Protection
## Stage 5 — Metadata & Data Catalog

This notebook extends the project into **Metadata Management and Data Cataloging**.

### Main Goals
1. Create a business glossary.
2. Build a technical metadata inventory.
3. Classify fields by data domain.
4. Assign proposed Data Owners and Data Stewards.
5. Define sensitivity and criticality.
6. Link metadata to Data Quality rules.
7. Link metadata to governance controls.
8. Identify Critical Data Elements (CDEs).
9. Create catalog-ready governance artifacts.

> Ownership, stewardship, sensitivity, and criticality assignments are proposed for this simulated CRM environment.


In [1]:
# 1. Libraries and Settings
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 300)

DATA_PATH = Path("Permission_Aware_CRM_Governance_Synthetic_50000.csv")
print("Environment ready.")


Environment ready.


# 2. Data Loading

In [2]:
df = pd.read_csv(DATA_PATH)
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
display(df.head())


Rows: 50,000
Columns: 15


,User_ID,Role,Region,Lead_Source,CRM_Action,Daily_Logins,Failed_Logins,Access_Hour,Device_Type,Data_Sensitivity,Policy_Compliance_Score,Anomaly_Score,Permission_Granted,Governance_Score,Access_Decision
0,1,Admin,North,Referral,ViewLead,6,1,10,Managed,5,76.19,0.114,True,51.81,Block
1,2,Sales Rep,West,Web,ViewLead,5,1,12,Managed,5,62.87,0.464,True,38.40,Block
2,3,Sales Rep,South,Partner,EditLead,6,1,20,Managed,1,81.87,0.083,False,48.57,Block
3,4,Analyst,South,Campaign,ViewLead,10,0,10,Managed,3,85.89,0.181,True,57.53,Block
4,5,Sales Rep,West,Web,EditLead,6,2,17,Managed,1,92.31,0.309,False,44.58,Block


# 3. Metadata Layers

### Technical Metadata
Physical representation: field name, data type, nullability, cardinality, example values.

### Business Metadata
Meaning and business context: definition, domain, business term, intended use.

### Governance Metadata
Controls and accountability: owner, steward, sensitivity, criticality, DQ rules, access rules.

### Operational Metadata
Source, refresh, lineage, transformation history.

This stage focuses mainly on technical, business, and governance metadata.


# 4. Technical Metadata Inventory

In [3]:
technical_metadata = pd.DataFrame({
    "Field_Name": df.columns,
    "Data_Type": [str(df[c].dtype) for c in df.columns],
    "Nullable": [df[c].isna().any() for c in df.columns],
    "Null_Count": [df[c].isna().sum() for c in df.columns],
    "Unique_Values": [df[c].nunique(dropna=True) for c in df.columns],
    "Cardinality_%": [df[c].nunique(dropna=True) / len(df) * 100 for c in df.columns],
    "Example_Value": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns]
})
display(technical_metadata.round(3))


,Field_Name,Data_Type,Nullable,Null_Count,Unique_Values,Cardinality_%,Example_Value
0,User_ID,int64,False,0,50000,100.000,1
1,Role,str,False,0,5,0.010,Admin
2,Region,str,False,0,4,0.008,North
3,Lead_Source,str,False,0,5,0.010,Referral
4,CRM_Action,str,False,0,6,0.012,ViewLead
5,Daily_Logins,int64,False,0,21,0.042,6
6,Failed_Logins,int64,False,0,7,0.014,1
7,Access_Hour,int64,False,0,17,0.034,10
8,Device_Type,str,False,0,2,0.004,Managed
9,Data_Sensitivity,int64,False,0,5,0.010,5


# 5. Business Glossary

In [4]:
business_glossary = pd.DataFrame([
    ["CRM User","An employee or system user who performs or requests actions within the CRM environment.","Identity & Access","User_ID, Role"],
    ["User Role","The functional role assigned to a CRM user and used as a baseline for role-based access control.","Identity & Access","Role"],
    ["CRM Action","A business or administrative operation requested or performed within the CRM.","CRM Operations","CRM_Action"],
    ["Data Sensitivity","An internal classification indicating the level of protection required for accessed information.","Data Governance","Data_Sensitivity"],
    ["Permission","An explicit authorization indicator showing whether the user has permission for an access request.","Identity & Access","Permission_Granted"],
    ["Access Decision","The resulting decision applied to an access request in the source synthetic dataset.","Access Governance","Access_Decision"],
    ["Policy Compliance Score","A score representing the level of compliance with defined access or governance policies.","Data Governance","Policy_Compliance_Score"],
    ["Anomaly Score","A numerical indicator representing how unusual an access event appears.","Security","Anomaly_Score"],
    ["Governance Score","An aggregated indicator representing governance-related characteristics of an access event.","Data Governance","Governance_Score"],
    ["Managed Device","A device managed under organizational controls and security policies.","Security","Device_Type"],
    ["BYOD","Bring Your Own Device; a user-owned device used to access organizational systems.","Security","Device_Type"]
], columns=["Business_Term","Definition","Domain","Related_Fields"])
display(business_glossary)


,Business_Term,Definition,Domain,Related_Fields
0,CRM User,An employee or system user who performs or req...,Identity & Access,"User_ID, Role"
1,User Role,The functional role assigned to a CRM user and...,Identity & Access,Role
2,CRM Action,A business or administrative operation request...,CRM Operations,CRM_Action
3,Data Sensitivity,An internal classification indicating the leve...,Data Governance,Data_Sensitivity
4,Permission,An explicit authorization indicator showing wh...,Identity & Access,Permission_Granted
5,Access Decision,The resulting decision applied to an access re...,Access Governance,Access_Decision
6,Policy Compliance Score,A score representing the level of compliance w...,Data Governance,Policy_Compliance_Score
7,Anomaly Score,A numerical indicator representing how unusual...,Security,Anomaly_Score
8,Governance Score,An aggregated indicator representing governanc...,Data Governance,Governance_Score
9,Managed Device,A device managed under organizational controls...,Security,Device_Type


# 6. Proposed Data Domains

In [5]:
field_domain_map = {
    "User_ID":"Identity & Access","Role":"Identity & Access","Region":"CRM Operations",
    "Lead_Source":"CRM Operations","CRM_Action":"CRM Operations","Daily_Logins":"Security",
    "Failed_Logins":"Security","Access_Hour":"Security","Device_Type":"Security",
    "Data_Sensitivity":"Data Governance","Policy_Compliance_Score":"Data Governance",
    "Anomaly_Score":"Security","Permission_Granted":"Identity & Access",
    "Governance_Score":"Data Governance","Access_Decision":"Access Governance"
}
domain_catalog = pd.DataFrame(field_domain_map.items(), columns=["Field_Name","Data_Domain"])
display(domain_catalog)


,Field_Name,Data_Domain
0,User_ID,Identity & Access
1,Role,Identity & Access
2,Region,CRM Operations
3,Lead_Source,CRM Operations
4,CRM_Action,CRM Operations
5,Daily_Logins,Security
6,Failed_Logins,Security
7,Access_Hour,Security
8,Device_Type,Security
9,Data_Sensitivity,Data Governance


# 7. Proposed Ownership and Stewardship

In [6]:
domain_governance = pd.DataFrame([
    ["Identity & Access","Identity & Access Management Lead","IAM Data Steward"],
    ["CRM Operations","CRM Business Owner","CRM Operations Data Steward"],
    ["Security","Information Security Lead","Security Data Steward"],
    ["Data Governance","Data Governance Lead","Data Governance Steward"],
    ["Access Governance","Access Governance Owner","Access Governance Steward"]
], columns=["Data_Domain","Proposed_Data_Owner","Proposed_Data_Steward"])
display(domain_governance)


,Data_Domain,Proposed_Data_Owner,Proposed_Data_Steward
0,Identity & Access,Identity & Access Management Lead,IAM Data Steward
1,CRM Operations,CRM Business Owner,CRM Operations Data Steward
2,Security,Information Security Lead,Security Data Steward
3,Data Governance,Data Governance Lead,Data Governance Steward
4,Access Governance,Access Governance Owner,Access Governance Steward


# 8. Sensitivity Classification

In [7]:
field_sensitivity_map = {
    "User_ID":"Restricted","Role":"Internal","Region":"Internal","Lead_Source":"Internal",
    "CRM_Action":"Confidential","Daily_Logins":"Confidential","Failed_Logins":"Restricted",
    "Access_Hour":"Confidential","Device_Type":"Confidential","Data_Sensitivity":"Restricted",
    "Policy_Compliance_Score":"Restricted","Anomaly_Score":"Restricted",
    "Permission_Granted":"Restricted","Governance_Score":"Restricted","Access_Decision":"Restricted"
}
field_sensitivity = pd.DataFrame(field_sensitivity_map.items(), columns=["Field_Name","Metadata_Sensitivity"])
display(field_sensitivity)


,Field_Name,Metadata_Sensitivity
0,User_ID,Restricted
1,Role,Internal
2,Region,Internal
3,Lead_Source,Internal
4,CRM_Action,Confidential
5,Daily_Logins,Confidential
6,Failed_Logins,Restricted
7,Access_Hour,Confidential
8,Device_Type,Confidential
9,Data_Sensitivity,Restricted


# 9. Critical Data Elements (CDEs)

In [8]:
criticality_map = {
    "User_ID":"Critical","Role":"Critical","Region":"Standard","Lead_Source":"Standard",
    "CRM_Action":"Critical","Daily_Logins":"Standard","Failed_Logins":"High",
    "Access_Hour":"High","Device_Type":"High","Data_Sensitivity":"Critical",
    "Policy_Compliance_Score":"High","Anomaly_Score":"High","Permission_Granted":"Critical",
    "Governance_Score":"High","Access_Decision":"Critical"
}
criticality = pd.DataFrame(criticality_map.items(), columns=["Field_Name","Criticality"])
criticality["Is_CDE"] = criticality["Criticality"].eq("Critical")
display(criticality)


,Field_Name,Criticality,Is_CDE
0,User_ID,Critical,True
1,Role,Critical,True
2,Region,Standard,False
3,Lead_Source,Standard,False
4,CRM_Action,Critical,True
5,Daily_Logins,Standard,False
6,Failed_Logins,High,False
7,Access_Hour,High,False
8,Device_Type,High,False
9,Data_Sensitivity,Critical,True


# 10. Business Definitions by Field

In [9]:
field_definitions = {
    "User_ID":"Identifier of the CRM user associated with the access event.",
    "Role":"Functional role assigned to the CRM user.",
    "Region":"Region associated with the CRM event or user context.",
    "Lead_Source":"Origin channel associated with the CRM lead.",
    "CRM_Action":"Action requested or performed in the CRM.",
    "Daily_Logins":"Number of login events associated with the user during the observed daily context.",
    "Failed_Logins":"Number of failed authentication attempts.",
    "Access_Hour":"Hour of day when the access event occurred.",
    "Device_Type":"Type of device used for CRM access.",
    "Data_Sensitivity":"Internal sensitivity level associated with the accessed data.",
    "Policy_Compliance_Score":"Score representing adherence to defined governance or access policies.",
    "Anomaly_Score":"Score representing the degree of anomalous access behavior.",
    "Permission_Granted":"Indicator showing whether explicit permission exists for the access request.",
    "Governance_Score":"Aggregated governance-related score associated with the access event.",
    "Access_Decision":"Final source decision assigned to the access event."
}
field_business_metadata = pd.DataFrame(field_definitions.items(), columns=["Field_Name","Business_Definition"])
display(field_business_metadata)


,Field_Name,Business_Definition
0,User_ID,Identifier of the CRM user associated with the...
1,Role,Functional role assigned to the CRM user.
2,Region,Region associated with the CRM event or user c...
3,Lead_Source,Origin channel associated with the CRM lead.
4,CRM_Action,Action requested or performed in the CRM.
5,Daily_Logins,Number of login events associated with the use...
6,Failed_Logins,Number of failed authentication attempts.
7,Access_Hour,Hour of day when the access event occurred.
8,Device_Type,Type of device used for CRM access.
9,Data_Sensitivity,Internal sensitivity level associated with the...


# 11. Source and System Metadata

In [10]:
source_metadata = pd.DataFrame({
    "Field_Name": df.columns,
    "Source_System":"Synthetic CRM Governance Dataset",
    "Source_Object":"Permission_Aware_CRM_Governance_Synthetic_50000.csv",
    "Source_Type":"CSV",
    "Refresh_Frequency":"Static / Project Dataset",
    "System_of_Record":"No — Synthetic Source"
})
display(source_metadata.head())


,Field_Name,Source_System,Source_Object,Source_Type,Refresh_Frequency,System_of_Record
0,User_ID,Synthetic CRM Governance Dataset,Permission_Aware_CRM_Governance_Synthetic_5000...,CSV,Static / Project Dataset,No — Synthetic Source
1,Role,Synthetic CRM Governance Dataset,Permission_Aware_CRM_Governance_Synthetic_5000...,CSV,Static / Project Dataset,No — Synthetic Source
2,Region,Synthetic CRM Governance Dataset,Permission_Aware_CRM_Governance_Synthetic_5000...,CSV,Static / Project Dataset,No — Synthetic Source
3,Lead_Source,Synthetic CRM Governance Dataset,Permission_Aware_CRM_Governance_Synthetic_5000...,CSV,Static / Project Dataset,No — Synthetic Source
4,CRM_Action,Synthetic CRM Governance Dataset,Permission_Aware_CRM_Governance_Synthetic_5000...,CSV,Static / Project Dataset,No — Synthetic Source


# 12. Link Fields to Data Quality Rules

In [11]:
dq_rule_field_map = pd.DataFrame([
    ["User_ID","DQ-COMP-001"],["Role","DQ-COMP-002"],["CRM_Action","DQ-COMP-003"],
    ["Permission_Granted","DQ-COMP-004"],["Access_Decision","DQ-COMP-005"],
    ["Role","DQ-VAL-001"],["CRM_Action","DQ-VAL-002"],["Device_Type","DQ-VAL-003"],
    ["Access_Hour","DQ-VAL-004"],["Data_Sensitivity","DQ-VAL-005"],
    ["Anomaly_Score","DQ-VAL-006"],["Policy_Compliance_Score","DQ-VAL-007"],
    ["Governance_Score","DQ-VAL-008"],["Daily_Logins","DQ-VAL-009"],
    ["Failed_Logins","DQ-VAL-010"],["Role","DQ-CON-001"],["CRM_Action","DQ-CON-001"],
    ["Permission_Granted","DQ-CON-002"],["Access_Decision","DQ-CON-002"]
], columns=["Field_Name","DQ_Rule_ID"])

dq_rules_by_field = (
    dq_rule_field_map.groupby("Field_Name")["DQ_Rule_ID"]
    .apply(lambda x: ", ".join(sorted(set(x))))
    .reset_index()
)
display(dq_rules_by_field)


,Field_Name,DQ_Rule_ID
0,Access_Decision,"DQ-COMP-005, DQ-CON-002"
1,Access_Hour,DQ-VAL-004
2,Anomaly_Score,DQ-VAL-006
3,CRM_Action,"DQ-COMP-003, DQ-CON-001, DQ-VAL-002"
4,Daily_Logins,DQ-VAL-009
5,Data_Sensitivity,DQ-VAL-005
6,Device_Type,DQ-VAL-003
7,Failed_Logins,DQ-VAL-010
8,Governance_Score,DQ-VAL-008
9,Permission_Granted,"DQ-COMP-004, DQ-CON-002"


# 13. Link Fields to Governance Controls

In [12]:
governance_rule_field_map = pd.DataFrame([
    ["Permission_Granted","AUTH-001"],["Role","AUTH-002"],["CRM_Action","AUTH-002"],
    ["Role","AUTH-003"],["CRM_Action","AUTH-003"],["Data_Sensitivity","CTX-001"],
    ["Device_Type","CTX-001"],["Failed_Logins","CTX-001"],["Anomaly_Score","CTX-001"],
    ["Policy_Compliance_Score","CTX-001"],["Governance_Score","CTX-001"],
    ["Data_Sensitivity","CTX-002"],["Device_Type","CTX-002"],["Anomaly_Score","CTX-002"],
    ["Data_Sensitivity","CTX-004"],["Failed_Logins","CTX-004"]
], columns=["Field_Name","Governance_Rule_ID"])

governance_rules_by_field = (
    governance_rule_field_map.groupby("Field_Name")["Governance_Rule_ID"]
    .apply(lambda x: ", ".join(sorted(set(x))))
    .reset_index()
)
display(governance_rules_by_field)


,Field_Name,Governance_Rule_ID
0,Anomaly_Score,"CTX-001, CTX-002"
1,CRM_Action,"AUTH-002, AUTH-003"
2,Data_Sensitivity,"CTX-001, CTX-002, CTX-004"
3,Device_Type,"CTX-001, CTX-002"
4,Failed_Logins,"CTX-001, CTX-004"
5,Governance_Score,CTX-001
6,Permission_Granted,AUTH-001
7,Policy_Compliance_Score,CTX-001
8,Role,"AUTH-002, AUTH-003"


# 14. Build the Master Data Catalog

In [13]:
data_catalog = (
    technical_metadata
    .merge(domain_catalog, on="Field_Name", how="left")
    .merge(domain_governance, on="Data_Domain", how="left")
    .merge(field_sensitivity, on="Field_Name", how="left")
    .merge(criticality, on="Field_Name", how="left")
    .merge(field_business_metadata, on="Field_Name", how="left")
    .merge(source_metadata, on="Field_Name", how="left")
    .merge(dq_rules_by_field, on="Field_Name", how="left")
    .merge(governance_rules_by_field, on="Field_Name", how="left")
)
display(data_catalog)


,Field_Name,Data_Type,Nullable,Null_Count,Unique_Values,Cardinality_%,Example_Value,Data_Domain,Proposed_Data_Owner,Proposed_Data_Steward,Metadata_Sensitivity,Criticality,Is_CDE,Business_Definition,Source_System,Source_Object,Source_Type,Refresh_Frequency,System_of_Record,DQ_Rule_ID,Governance_Rule_ID
0,User_ID,int64,False,0,50000,100.000,1,Identity & Access,Identity & Access Management Lead,IAM Data Steward,Restricted,Critical,True,Identifier of the CRM user associated with the...,Synthetic CRM Governance Dataset,Permission_Aware_CRM_Governance_Synthetic_5000...,CSV,Static / Project Dataset,No — Synthetic Source,DQ-COMP-001,NaN
1,Role,str,False,0,5,0.010,Admin,Identity & Access,Identity & Access Management Lead,IAM Data Steward,Internal,Critical,True,Functional role assigned to the CRM user.,Synthetic CRM Governance Dataset,Permission_Aware_CRM_Governance_Synthetic_5000...,CSV,Static / Project Dataset,No — Synthetic Source,"DQ-COMP-002, DQ-CON-001, DQ-VAL-001","AUTH-002, AUTH-003"
2,Region,str,False,0,4,0.008,North,CRM Operations,CRM Business Owner,CRM Operations Data Steward,Internal,Standard,False,Region associated with the CRM event or user c...,Synthetic CRM Governance Dataset,Permission_Aware_CRM_Governance_Synthetic_5000...,CSV,Static / Project Dataset,No — Synthetic Source,NaN,NaN
3,Lead_Source,str,False,0,5,0.010,Referral,CRM Operations,CRM Business Owner,CRM Operations Data Steward,Internal,Standard,False,Origin channel associated with the CRM lead.,Synthetic CRM Governance Dataset,Permission_Aware_CRM_Governance_Synthetic_5000...,CSV,Static / Project Dataset,No — Synthetic Source,NaN,NaN
4,CRM_Action,str,False,0,6,0.012,ViewLead,CRM Operations,CRM Business Owner,CRM Operations Data Steward,Confidential,Critical,True,Action requested or performed in the CRM.,Synthetic CRM Governance Dataset,Permission_Aware_CRM_Governance_Synthetic_5000...,CSV,Static / Project Dataset,No — Synthetic Source,"DQ-COMP-003, DQ-CON-001, DQ-VAL-002","AUTH-002, AUTH-003"
5,Daily_Logins,int64,False,0,21,0.042,6,Security,Information Security Lead,Security Data Steward,Confidential,Standard,False,Number of login events associated with the use...,Synthetic CRM Governance Dataset,Permission_Aware_CRM_Governance_Synthetic_5000...,CSV,Static / Project Dataset,No — Synthetic Source,DQ-VAL-009,NaN
6,Failed_Logins,int64,False,0,7,0.014,1,Security,Information Security Lead,Security Data Steward,Restricted,High,False,Number of failed authentication attempts.,Synthetic CRM Governance Dataset,Permission_Aware_CRM_Governance_Synthetic_5000...,CSV,Static / Project Dataset,No — Synthetic Source,DQ-VAL-010,"CTX-001, CTX-004"
7,Access_Hour,int64,False,0,17,0.034,10,Security,Information Security Lead,Security Data Steward,Confidential,High,False,Hour of day when the access event occurred.,Synthetic CRM Governance Dataset,Permission_Aware_CRM_Governance_Synthetic_5000...,CSV,Static / Project Dataset,No — Synthetic Source,DQ-VAL-004,NaN
8,Device_Type,str,False,0,2,0.004,Managed,Security,Information Security Lead,Security Data Steward,Confidential,High,False,Type of device used for CRM access.,Synthetic CRM Governance Dataset,Permission_Aware_CRM_Governance_Synthetic_5000...,CSV,Static / Project Dataset,No — Synthetic Source,DQ-VAL-003,"CTX-001, CTX-002"
9,Data_Sensitivity,int64,False,0,5,0.010,5,Data Governance,Data Governance Lead,Data Governance Steward,Restricted,Critical,True,Internal sensitivity level associated with the...,Synthetic CRM Governance Dataset,Permission_Aware_CRM_Governance_Synthetic_5000...,CSV,Static / Project Dataset,No — Synthetic Source,DQ-VAL-005,"CTX-001, CTX-002, CTX-004"


# 15. Catalog Completeness Check

In [14]:
required_catalog_fields = [
    "Field_Name","Data_Type","Business_Definition","Data_Domain",
    "Proposed_Data_Owner","Proposed_Data_Steward","Metadata_Sensitivity",
    "Criticality","Source_System"
]

catalog_completeness = pd.DataFrame({
    "Metadata_Field": required_catalog_fields,
    "Missing_Count": [data_catalog[c].isna().sum() for c in required_catalog_fields],
    "Completeness_%": [data_catalog[c].notna().mean()*100 for c in required_catalog_fields]
})
display(catalog_completeness.round(2))


,Metadata_Field,Missing_Count,Completeness_%
0,Field_Name,0,100.0
1,Data_Type,0,100.0
2,Business_Definition,0,100.0
3,Data_Domain,0,100.0
4,Proposed_Data_Owner,0,100.0
5,Proposed_Data_Steward,0,100.0
6,Metadata_Sensitivity,0,100.0
7,Criticality,0,100.0
8,Source_System,0,100.0


# 16. Critical Data Element Inventory

In [15]:
cde_inventory = data_catalog[data_catalog["Is_CDE"]][[
    "Field_Name","Business_Definition","Data_Domain","Proposed_Data_Owner",
    "Proposed_Data_Steward","Metadata_Sensitivity","Criticality",
    "DQ_Rule_ID","Governance_Rule_ID"
]]
display(cde_inventory)


,Field_Name,Business_Definition,Data_Domain,Proposed_Data_Owner,Proposed_Data_Steward,Metadata_Sensitivity,Criticality,DQ_Rule_ID,Governance_Rule_ID
0,User_ID,Identifier of the CRM user associated with the...,Identity & Access,Identity & Access Management Lead,IAM Data Steward,Restricted,Critical,DQ-COMP-001,NaN
1,Role,Functional role assigned to the CRM user.,Identity & Access,Identity & Access Management Lead,IAM Data Steward,Internal,Critical,"DQ-COMP-002, DQ-CON-001, DQ-VAL-001","AUTH-002, AUTH-003"
4,CRM_Action,Action requested or performed in the CRM.,CRM Operations,CRM Business Owner,CRM Operations Data Steward,Confidential,Critical,"DQ-COMP-003, DQ-CON-001, DQ-VAL-002","AUTH-002, AUTH-003"
9,Data_Sensitivity,Internal sensitivity level associated with the...,Data Governance,Data Governance Lead,Data Governance Steward,Restricted,Critical,DQ-VAL-005,"CTX-001, CTX-002, CTX-004"
12,Permission_Granted,Indicator showing whether explicit permission ...,Identity & Access,Identity & Access Management Lead,IAM Data Steward,Restricted,Critical,"DQ-COMP-004, DQ-CON-002",AUTH-001
14,Access_Decision,Final source decision assigned to the access e...,Access Governance,Access Governance Owner,Access Governance Steward,Restricted,Critical,"DQ-COMP-005, DQ-CON-002",NaN


# 17. Metadata Sensitivity Inventory

In [16]:
sensitivity_inventory = (
    data_catalog.groupby("Metadata_Sensitivity")
    .agg(Fields=("Field_Name","count"), Critical_Data_Elements=("Is_CDE","sum"))
    .reset_index()
)
display(sensitivity_inventory)


,Metadata_Sensitivity,Fields,Critical_Data_Elements
0,Confidential,4,1
1,Internal,3,1
2,Restricted,8,4


# 18. Domain Inventory

In [17]:
domain_inventory = (
    data_catalog.groupby(["Data_Domain","Proposed_Data_Owner","Proposed_Data_Steward"], dropna=False)
    .agg(Fields=("Field_Name","count"), Critical_Data_Elements=("Is_CDE","sum"))
    .reset_index()
)
display(domain_inventory)


,Data_Domain,Proposed_Data_Owner,Proposed_Data_Steward,Fields,Critical_Data_Elements
0,Access Governance,Access Governance Owner,Access Governance Steward,1,1
1,CRM Operations,CRM Business Owner,CRM Operations Data Steward,3,1
2,Data Governance,Data Governance Lead,Data Governance Steward,3,1
3,Identity & Access,Identity & Access Management Lead,IAM Data Steward,3,3
4,Security,Information Security Lead,Security Data Steward,5,0


# 19. Data Catalog Views

In [18]:
business_view = data_catalog[[
    "Field_Name","Business_Definition","Data_Domain","Proposed_Data_Owner","Proposed_Data_Steward"
]]

technical_view = data_catalog[[
    "Field_Name","Data_Type","Nullable","Null_Count","Unique_Values","Cardinality_%",
    "Source_System","Source_Object"
]]

governance_view = data_catalog[[
    "Field_Name","Metadata_Sensitivity","Criticality","Is_CDE","DQ_Rule_ID","Governance_Rule_ID"
]]

display(business_view)
display(technical_view.round(3))
display(governance_view)


,Field_Name,Business_Definition,Data_Domain,Proposed_Data_Owner,Proposed_Data_Steward
0,User_ID,Identifier of the CRM user associated with the...,Identity & Access,Identity & Access Management Lead,IAM Data Steward
1,Role,Functional role assigned to the CRM user.,Identity & Access,Identity & Access Management Lead,IAM Data Steward
2,Region,Region associated with the CRM event or user c...,CRM Operations,CRM Business Owner,CRM Operations Data Steward
3,Lead_Source,Origin channel associated with the CRM lead.,CRM Operations,CRM Business Owner,CRM Operations Data Steward
4,CRM_Action,Action requested or performed in the CRM.,CRM Operations,CRM Business Owner,CRM Operations Data Steward
5,Daily_Logins,Number of login events associated with the use...,Security,Information Security Lead,Security Data Steward
6,Failed_Logins,Number of failed authentication attempts.,Security,Information Security Lead,Security Data Steward
7,Access_Hour,Hour of day when the access event occurred.,Security,Information Security Lead,Security Data Steward
8,Device_Type,Type of device used for CRM access.,Security,Information Security Lead,Security Data Steward
9,Data_Sensitivity,Internal sensitivity level associated with the...,Data Governance,Data Governance Lead,Data Governance Steward


,Field_Name,Data_Type,Nullable,Null_Count,Unique_Values,Cardinality_%,Source_System,Source_Object
0,User_ID,int64,False,0,50000,100.000,Synthetic CRM Governance Dataset,Permission_Aware_CRM_Governance_Synthetic_5000...
1,Role,str,False,0,5,0.010,Synthetic CRM Governance Dataset,Permission_Aware_CRM_Governance_Synthetic_5000...
2,Region,str,False,0,4,0.008,Synthetic CRM Governance Dataset,Permission_Aware_CRM_Governance_Synthetic_5000...
3,Lead_Source,str,False,0,5,0.010,Synthetic CRM Governance Dataset,Permission_Aware_CRM_Governance_Synthetic_5000...
4,CRM_Action,str,False,0,6,0.012,Synthetic CRM Governance Dataset,Permission_Aware_CRM_Governance_Synthetic_5000...
5,Daily_Logins,int64,False,0,21,0.042,Synthetic CRM Governance Dataset,Permission_Aware_CRM_Governance_Synthetic_5000...
6,Failed_Logins,int64,False,0,7,0.014,Synthetic CRM Governance Dataset,Permission_Aware_CRM_Governance_Synthetic_5000...
7,Access_Hour,int64,False,0,17,0.034,Synthetic CRM Governance Dataset,Permission_Aware_CRM_Governance_Synthetic_5000...
8,Device_Type,str,False,0,2,0.004,Synthetic CRM Governance Dataset,Permission_Aware_CRM_Governance_Synthetic_5000...
9,Data_Sensitivity,int64,False,0,5,0.010,Synthetic CRM Governance Dataset,Permission_Aware_CRM_Governance_Synthetic_5000...


,Field_Name,Metadata_Sensitivity,Criticality,Is_CDE,DQ_Rule_ID,Governance_Rule_ID
0,User_ID,Restricted,Critical,True,DQ-COMP-001,NaN
1,Role,Internal,Critical,True,"DQ-COMP-002, DQ-CON-001, DQ-VAL-001","AUTH-002, AUTH-003"
2,Region,Internal,Standard,False,NaN,NaN
3,Lead_Source,Internal,Standard,False,NaN,NaN
4,CRM_Action,Confidential,Critical,True,"DQ-COMP-003, DQ-CON-001, DQ-VAL-002","AUTH-002, AUTH-003"
5,Daily_Logins,Confidential,Standard,False,DQ-VAL-009,NaN
6,Failed_Logins,Restricted,High,False,DQ-VAL-010,"CTX-001, CTX-004"
7,Access_Hour,Confidential,High,False,DQ-VAL-004,NaN
8,Device_Type,Confidential,High,False,DQ-VAL-003,"CTX-001, CTX-002"
9,Data_Sensitivity,Restricted,Critical,True,DQ-VAL-005,"CTX-001, CTX-002, CTX-004"


# 20. Proposed Metadata Governance Operating Model

| Activity | Proposed Responsibility |
|---|---|
| Approve business definitions | Data Owner |
| Maintain glossary and catalog metadata | Data Steward |
| Maintain technical metadata | Data Engineering |
| Maintain sensitivity classification | Data Governance + Security |
| Maintain privacy classification | Privacy / DPO |
| Approve Critical Data Elements | Data Governance Council / Data Owner |
| Monitor catalog completeness | Data Governance Team |


# 21. Catalog KPIs

Future governance monitoring can include:
- Catalog Coverage %
- Business Definition Completeness %
- Ownership Assignment %
- Stewardship Assignment %
- Sensitivity Classification Coverage %
- CDE Coverage %
- Fields with Data Quality Rules
- Fields with Governance Rules
- Assets by Domain
- Restricted Fields


# 22. Exportable Metadata Artifacts

Reusable artifacts:
- `business_glossary`
- `technical_metadata`
- `domain_catalog`
- `domain_governance`
- `field_sensitivity`
- `criticality`
- `dq_rule_field_map`
- `governance_rule_field_map`
- `data_catalog`
- `cde_inventory`


# 23. Findings to Document


## Catalog Coverage

- **Total cataloged fields:** 15 fields are included in the master data catalog.

- **Metadata completeness:** 100% for all required catalog metadata attributes evaluated, including field name, data type, business definition, data domain, proposed owner, proposed steward, sensitivity classification, criticality, and source system.

- **Missing ownership assignments:** 0. All 15 cataloged fields have a proposed Data Owner assigned.

- **Missing stewardship assignments:** 0. All 15 cataloged fields have a proposed Data Steward assigned.

## Data Domains

- **Number of domains:** 5 — `Identity & Access`, `CRM Operations`, `Security`, `Data Governance`, and `Access Governance`.

- **Largest domain:** `Security`, with 5 cataloged fields.

- **Domains with the most CDEs:** `Identity & Access` has the highest number of Critical Data Elements, with 3 CDEs. `CRM Operations`, `Data Governance`, and `Access Governance` each contain 1 CDE, while `Security` contains none.

## Critical Data Elements

- **Total CDEs:** 6.

- **CDEs by domain:**
  - `Identity & Access`: 3
  - `CRM Operations`: 1
  - `Data Governance`: 1
  - `Access Governance`: 1
  - `Security`: 0

- **CDEs with DQ rules:** 6 of 6 CDEs (100%) are linked to at least one Data Quality rule.

- **CDEs with governance controls:** 4 of 6 CDEs are explicitly linked to governance rules in the current catalog. `User_ID` and `Access_Decision` currently have no `Governance_Rule_ID` assigned.

## Sensitivity

- **Restricted fields:** 8
- **Confidential fields:** 4
- **Internal fields:** 3

## Governance Conclusions

1. **The catalog achieves complete metadata coverage for the 15 fields in scope, with business definitions, domains, proposed ownership, stewardship, sensitivity, criticality, and source information fully documented.**

2. **Critical Data Elements are concentrated in the Identity & Access domain and all CDEs are connected to Data Quality controls, establishing strong traceability between critical metadata and measurable quality requirements.**

3. **Governance-control traceability is not yet complete for all CDEs: four of the six CDEs are explicitly linked to governance rules, while `User_ID` and `Access_Decision` remain without a direct governance-rule mapping and should be evaluated in later governance and lineage stages.**

# 24. Limitations

1. The catalog covers one synthetic dataset.
2. Ownership and stewardship assignments are simulated.
3. Sensitivity labels are internal project classifications.
4. There is no automated metadata scanner.
5. There is no catalog platform integration yet.
6. Operational metadata and lineage events are not available.
7. Customer-level PII is not present in the current dataset.


# 25. Next Step — Data Classification & Privacy Engineering

## Stage 6 — Data Classification & Privacy Engineering

Planned outputs:
- simulated customer data layer;
- field-level PII classification;
- direct vs. indirect identifiers;
- data minimization rules;
- masking logic;
- pseudonymization/tokenization;
- analytics-safe customer layer;
- privacy control catalog;
- mapping between privacy controls and access governance.
